# Cross-Modal Diagnostic Observability — Stage 11D-R

## tl;dr

This outcome-free notebook turns the five qualified Stage 11C-R2 breast-ultrasound receipts into an auditable image–label–group manifest, removes cross-roster exact and high-confidence derived-copy candidates, and freezes deterministic patient/lesion-grouped held-out and OOF partitions.

## Context & Methods

Stage 11C-R2 established that all five official receipts expose pixels, defensible labels, and deterministic patient/lesion groups. Stage 11D-R now addresses the next publication-critical risks: endpoint leakage, masks treated as images, duplicated pixels or derived copies across public datasets, and patient leakage across model-development partitions.

### Key assumptions and boundaries

- The frozen breast-ultrasound endpoint is malignant lesion versus benign lesion. Normal, unknown, indeterminate, missing, or BI-RADS-only rows are excluded.
- Exact duplicates are resolved by the frozen Stage 11C-R2 roster priority. Label-conflicting exact clusters are quarantined in full.
- High-confidence near-copy candidates use an outcome-free image-only rule and are conservatively quarantined before splitting; they are not called biological duplicates.
- All images from one released patient/lesion group remain in one held-out/development partition and one OOF fold.
- No embedding, source axis, AUC, transfer result, threshold, DDO2 fit, Stage 12 operation, or locked-blind asset is accessed.


In [1]:
# @title 11D-R-0. Mount Drive, verify sealed parents, and freeze the pre-performance protocol
import hashlib, io, json, os, re, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from scipy.fft import dctn

try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    pass

DEFAULT_ROOT = Path('/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability') if IN_COLAB else Path('/tmp/Cross-Modal_Diagnostic_Observability')
PROJECT_ROOT = Path(os.environ.get('CDO_PROJECT_ROOT', str(DEFAULT_ROOT)))
CODE_ROOT = PROJECT_ROOT/'05_Code'/'Cross_Modal'
CM_ROOT = PROJECT_ROOT/'06_Data_Records'/'Cross_Modal'
STAGE10_ROOT = CM_ROOT/'Stage10_Preassigned_Development_And_Locked_Blind_Registry_v0.1'
R2_ROOT = CM_ROOT/'Stage11C-R2_Recursive_Payload_And_Provider_Evidence_Readjudication_v0.1'
ROOT = CM_ROOT/'Stage11D-R_Cross_Roster_Dedup_Exact_Manifest_And_Grouped_Split_Freeze_v0.1'
RECEIPT_ROOT = PROJECT_ROOT/'00_Data_Acquisition'/'Stage11C_Manual_Official_Receipts'
P0,P1,P2,P3,P4,P5 = [ROOT/x for x in ['00_Protocol','01_Exact_Manifest','02_Deduplication','03_Grouped_Splits','04_Firewall','05_Results']]
for p in [CODE_ROOT,P0,P1,P2,P3,P4,P5]: p.mkdir(parents=True,exist_ok=True)

NOTEBOOK_NAME='CrossModal_Stage11D-R_Cross_Roster_Dedup_Exact_Manifest_And_Grouped_Split_Freeze_v0.1.ipynb'
NOTEBOOK_PATH=CODE_ROOT/NOTEBOOK_NAME
STAGE10_FINAL=STAGE10_ROOT/'05_Results'/'Stage10_Preassigned_Role_Registry_Complete_v0.1.json'
ENDPOINT_MAP=STAGE10_ROOT/'02_Endpoint_And_Access_Audit'/'Stage10_Frozen_Label_Mapping_And_Grouping_Audit_v0.1.csv'
OVERLAP_GATES=STAGE10_ROOT/'02_Endpoint_And_Access_Audit'/'Stage10_Frozen_Dataset_Overlap_Gates_v0.1.csv'
R2_FINAL=R2_ROOT/'05_Results'/'Stage11C-R2_Complete_v0.1.json'
R2_HANDOFF=R2_ROOT/'03_Readjudication'/'Stage11C-R2_Stage11D-R_Handoff_v0.1.json'
R2_ROSTER=R2_ROOT/'03_Readjudication'/'Stage11C-R2_Qualified_Development_Roster_v0.1.csv'
R2_RECEIPTS=R2_ROOT/'01_Recursive_Inventory'/'Stage11C-R2_Receipt_Reverification_v0.1.csv'

EXPECTED_STAGE10_FINAL='6438434cb41607ad97b1b4a2fab07b143969ce7f551b87b5c2a23afbe67ccccf'
EXPECTED_R2_FINAL='fffcb9dba5c5e25c066b5f92cdd3856bbe84dbe42eb1ae0cab62a9228c70f85d'
EXPECTED_R2_HANDOFF='24da138b2a6972fe60108902bf7a03d6523b2fff47abbfcbe1212feb7b8a9801'
DATASETS=['BUS_BRA_2024','BUSI_WHU_2025_V3','BREAST_LESIONS_USG_2024','BUS_UCLM_2025_V3','RODRIGUES_BUI_2017']
FOLDERS={'BUS_BRA_2024':'BUS_BRA','BUSI_WHU_2025_V3':'BUSI_WHU','BREAST_LESIONS_USG_2024':'BREAST_LESIONS_USG','BUS_UCLM_2025_V3':'BUS_UCLM','RODRIGUES_BUI_2017':'RODRIGUES_BUI'}
EXPECTED_ORIGINALS={'BUS_BRA_2024':1875,'BUSI_WHU_2025_V3':927,'BREAST_LESIONS_USG_2024':256,'BUS_UCLM_2025_V3':683,'RODRIGUES_BUI_2017':250}
EXPECTED_GROUPS={'BUS_BRA_2024':1064,'BUSI_WHU_2025_V3':816,'BREAST_LESIONS_USG_2024':256,'BUS_UCLM_2025_V3':38,'RODRIGUES_BUI_2017':250}
LOCKED=['BUSI_CAIRO_2019','OASBUD_2017','DERM7PT_2019']
SEED=20260721
HOLDOUT_FRACTION=0.20
MAX_OOF_FOLDS=5
MIN_OOF_FOLDS=3
MIN_GROUPS=12
PHASH_MAX_DISTANCE=4
NEAR_COPY_MIN_CORRELATION=0.995
MINIMUM_SPLIT_READY_DOMAINS=4

PROTOCOL=P0/'Stage11D-R_Protocol_Seal_v0.1.json'
PARENT_COMMIT=P0/'Stage11D-R_Parent_Input_Commitment_v0.1.csv'
ADAPTER_STATUS=P1/'Stage11D-R_Dataset_Adapter_Status_v0.1.csv'
ALL_MANIFEST=P1/'Stage11D-R_All_Source_Image_Label_Group_Manifest_v0.1.csv'
EXACT_MANIFEST=P1/'Stage11D-R_Frozen_Exact_Image_Label_Group_Manifest_v0.1.csv'
EXACT_CLUSTERS=P2/'Stage11D-R_Exact_Pixel_Duplicate_Clusters_v0.1.csv'
NEAR_CANDIDATES=P2/'Stage11D-R_High_Confidence_Near_Copy_Candidates_v0.1.csv'
DEDUP_SUMMARY=P2/'Stage11D-R_Deduplication_And_Endpoint_Summary_v0.1.csv'
GROUP_ASSIGN=P3/'Stage11D-R_Frozen_Group_Assignments_v0.1.csv'
SPLIT_MANIFEST=P3/'Stage11D-R_Frozen_Grouped_Split_Manifest_v0.1.csv'
SPLIT_SUMMARY=P3/'Stage11D-R_Grouped_Split_Summary_v0.1.csv'
HANDOFF=P3/'Stage11D-R_Stage11E-R_Handoff_v0.1.json'
FIREWALL=P4/'Stage11D-R_Independent_Validity_And_Firewall_Checks_v0.1.csv'
REPORT=P5/'Stage11D-R_Dedup_Manifest_And_Grouped_Split_Report_v0.1.md'
OUTPUT_MANIFEST=P5/'Stage11D-R_Output_Integrity_Manifest_v0.1.csv'
FINAL=P5/'Stage11D-R_Complete_v0.1.json'
RUNTIME=P5/'Stage11D-R_Runtime_State_v0.1.json'

def now(): return datetime.now(timezone.utc).isoformat()
def sha_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def sha_bytes(b): return hashlib.sha256(b).hexdigest()
def sha_json(x): return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def canon(df): return df.fillna('').to_csv(index=False,lineterminator='\n',float_format='%.12g')
def write_text(p,s):
    p=Path(p)
    if p.exists(): assert p.read_text(encoding='utf-8')==s, f'Replay mismatch: {p}'
    else: p.write_text(s,encoding='utf-8')
def write_csv(p,d): write_text(p,canon(d))
def write_json(p,x): write_text(p,json.dumps(x,indent=2,ensure_ascii=False)+'\n')
def verify_self(p,field,expected=None):
    x=json.loads(Path(p).read_text(encoding='utf-8')); claimed=x[field]; y=dict(x); y.pop(field)
    assert sha_json(y)==claimed,f'Self-hash mismatch: {p}'
    if expected: assert claimed==expected,f'Unexpected hash: {p}'
    return x
def markdown_table(df):
    x=df.fillna('').astype(str); esc=lambda s:str(s).replace('|','\\|').replace('\n',' ')
    return '\n'.join(['| '+' | '.join(esc(c) for c in x.columns)+' |','| '+' | '.join('---' for _ in x.columns)+' |']+['| '+' | '.join(esc(v) for v in row)+' |' for row in x.itertuples(index=False,name=None)])

required=[NOTEBOOK_PATH,STAGE10_FINAL,ENDPOINT_MAP,OVERLAP_GATES,R2_FINAL,R2_HANDOFF,R2_ROSTER,R2_RECEIPTS]
missing=[str(p) for p in required if not p.is_file()]
assert not missing,'Missing sealed inputs:\n'+'\n'.join(missing)
stage10=verify_self(STAGE10_FINAL,'final_record_sha256',EXPECTED_STAGE10_FINAL)
r2=verify_self(R2_FINAL,'final_record_sha256',EXPECTED_R2_FINAL)
r2_handoff=verify_self(R2_HANDOFF,'handoff_sha256',EXPECTED_R2_HANDOFF)
assert r2['stage11d_r_authorised'] is True and r2_handoff['stage11d_r_authorised'] is True
assert r2_handoff['qualified_dataset_ids']==DATASETS and r2['locked_blind_assets_touched'] is False
assert sha_file(R2_ROSTER)==r2['qualified_roster_sha256'] and sha_file(R2_RECEIPTS)==r2['receipt_reverification_sha256']
roster=pd.read_csv(R2_ROSTER); receipts=pd.read_csv(R2_RECEIPTS); endpoint_map=pd.read_csv(ENDPOINT_MAP); overlap_gates=pd.read_csv(OVERLAP_GATES)
assert roster.dataset_id.tolist()==DATASETS
breast_endpoint=endpoint_map[(endpoint_map.modality=='breast_ultrasound')&(endpoint_map.task=='breast_lesion_malignant_vs_benign')]
assert len(breast_endpoint)==1 and 'normal' in str(breast_endpoint.iloc[0].excluded).lower()
assert stage10['endpoint_specification_sha256']==sha_file(STAGE10_ROOT/'01_Role_Registry'/'Stage10_Frozen_Harmonised_Endpoint_Specifications_v0.1.json')
REPLAY=FINAL.is_file()

commit=pd.DataFrame([{'role':p.name,'relative_path':str(p.relative_to(PROJECT_ROOT)),'size_bytes':p.stat().st_size,'sha256':sha_file(p)} for p in required[1:]])
if REPLAY: assert canon(pd.read_csv(PARENT_COMMIT))==canon(commit)
else: write_csv(PARENT_COMMIT,commit)
spec={
 'scope':'FIVE_QUALIFIED_DEVELOPMENT_RECEIPTS_ENDPOINT_HARMONISATION_DEDUP_AND_GROUPED_SPLIT_ONLY',
 'endpoint':'released malignant lesion = 1; released benign lesion = 0; normal/unknown/indeterminate/missing/BI-RADS-only excluded',
 'exact_duplicate_policy':'decoded RGB pixel SHA-256; conflicting-label clusters quarantine all; otherwise keep frozen-roster-priority canonical row',
 'near_copy_policy':{'phash_max_distance':PHASH_MAX_DISTANCE,'thumbnail_correlation_minimum':NEAR_COPY_MIN_CORRELATION,'resolution':'conservatively quarantine all but frozen-priority canonical row; quarantine all on label conflict'},
 'split_policy':{'seed':SEED,'heldout_fraction':HOLDOUT_FRACTION,'max_oof_folds':MAX_OOF_FOLDS,'minimum_oof_folds':MIN_OOF_FOLDS,'minimum_groups':MIN_GROUPS,'group_unit':'released patient; released lesion only where no patient key exists'},
 'prohibited':['embeddings','source axis','AUC','transfer outcome','threshold selection','DDO2','Stage12','locked-blind assets']}
payload={'stage':'Stage11D-R','version':'0.1','parent_stage10_final_sha256':stage10['final_record_sha256'],'parent_stage11c_r2_final_sha256':r2['final_record_sha256'],'parent_stage11c_r2_handoff_sha256':r2_handoff['handoff_sha256'],'roster_sha256':sha_file(R2_ROSTER),'receipt_reverification_sha256':sha_file(R2_RECEIPTS),'endpoint_map_sha256':sha_file(ENDPOINT_MAP),'overlap_gate_sha256':sha_file(OVERLAP_GATES),'analysis_spec':spec}
if REPLAY:
    seal=verify_self(PROTOCOL,'seal_sha256')
    for k,v in payload.items(): assert seal[k]==v
else:
    seal=dict(payload); seal['sealed_utc']=now(); seal['seal_sha256']=sha_json(seal); write_json(PROTOCOL,seal)
runtime={'stage':'Stage11D-R','replay_mode':REPLAY,'performance_evaluated':False,'embeddings_computed':False,'source_axes_fitted':False,'transfer_edges_evaluated':False,'ddo2_fitted':False,'stage12_authorised':False,'locked_blind_assets_touched':False}
print('Stage10 / Stage11C-R2 / handoff verified:',stage10['final_record_sha256'],r2['final_record_sha256'],r2_handoff['handoff_sha256'])
print('Protocol seal / Replay:',seal['seal_sha256'],REPLAY)


Mounted at /content/drive
Stage10 / Stage11C-R2 / handoff verified: 6438434cb41607ad97b1b4a2fab07b143969ce7f551b87b5c2a23afbe67ccccf fffcb9dba5c5e25c066b5f92cdd3856bbe84dbe42eb1ae0cab62a9228c70f85d 24da138b2a6972fe60108902bf7a03d6523b2fff47abbfcbe1212feb7b8a9801
Protocol seal / Replay: 8f2c409cf51429c33aea93b0c8788484c53c909865c4153ca88de3bb3877cc98 False


In [2]:
# @title 11D-R-1. Build exact image–label–group rows from the five sealed receipts
IMAGE_EXT={'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}
zip_handles=[]; zip_buffers=[]; entry_lookup={}; outer_paths={}

def norm(s): return re.sub(r'[^a-z0-9]+','_',str(s).strip().lower()).strip('_')
def compact(s): return re.sub(r'[^a-z0-9]+','',str(s).strip().lower())
def stable_value(v):
    if pd.isna(v): return ''
    s=str(v).strip()
    return re.sub(r'\.0$','',s) if re.fullmatch(r'[-+]?\d+\.0',s) else s
def variants(v):
    s=stable_value(v); stem=Path(s).stem; out={compact(s),compact(stem)}
    nums=re.findall(r'\d+',stem)
    if nums:
        n='-'.join(str(int(x)) for x in nums); out|={compact(n),compact('case'+n),compact('bus'+n)}
    return {x for x in out if x}
def path_leaf(v): return PurePosixPath(str(v).split('!/')[-1]).name
def path_stem(v): return PurePosixPath(str(v).split('!/')[-1]).stem
def is_dir(v,name): return bool(re.search(r'(^|[/!])'+re.escape(name)+r'([/!]|$)',str(v),flags=re.I))
def image_entries(d): return sorted([v for (dd,v),e in entry_lookup.items() if dd==d and e['suffix'] in IMAGE_EXT])
def read_virtual(d,v):
    e=entry_lookup[(d,v)]; return e['zip'].read(e['member'])
def register_zip(d,outer,z,prefix=(),depth=0):
    zip_handles.append(z)
    for info in z.infolist():
        if info.is_dir(): continue
        chain=tuple(prefix)+(info.filename,); virtual='!/'.join(chain); suffix=Path(info.filename).suffix.lower()
        entry_lookup[(d,virtual)]={'zip':z,'member':info.filename,'suffix':suffix,'size':int(info.file_size),'outer':outer.name,'depth':depth}
        if suffix=='.zip':
            b=z.read(info); buf=io.BytesIO(b); zip_buffers.append(buf); register_zip(d,outer,zipfile.ZipFile(buf),chain,depth+1)

expected_receipts={(r.dataset_id,r.file_name):(int(r.size_bytes),str(r.sha256)) for r in receipts.itertuples()}
observed_receipts=[]
for d in DATASETS:
    folder=RECEIPT_ROOT/FOLDERS[d]
    files=sorted([p for p in folder.iterdir() if p.is_file() and not p.name.startswith('.')]) if folder.is_dir() else []
    outer_paths[d]=files
    for p in files:
        key=(d,p.name); h=sha_file(p); size=p.stat().st_size
        observed_receipts.append({'dataset_id':d,'file_name':p.name,'size_bytes':size,'sha256':h,'matches_stage11c_r2':key in expected_receipts and expected_receipts[key]==(size,h)})
        if p.suffix.lower()=='.zip': register_zip(d,p,zipfile.ZipFile(p))
observed_receipts=pd.DataFrame(observed_receipts)
assert len(observed_receipts)==len(expected_receipts) and observed_receipts.matches_stage11c_r2.all(),'Receipt bytes changed after Stage11C-R2'

def read_table_entry(d,predicate):
    candidates=[]
    for (dd,v),e in entry_lookup.items():
        if dd!=d or not predicate(v): continue
        b=read_virtual(d,v); suf=e['suffix']
        if suf=='.csv':
            for enc in ['utf-8-sig','utf-8','latin1']:
                try: candidates.append((v,'table',pd.read_csv(io.BytesIO(b),encoding=enc))); break
                except Exception: pass
        elif suf in {'.xlsx','.xls'}:
            book=pd.ExcelFile(io.BytesIO(b)); candidates.extend((v,sh,pd.read_excel(io.BytesIO(b),sheet_name=sh)) for sh in book.sheet_names)
    return candidates
def pick_col(frame,patterns,exact=()):
    exact_norm={norm(x) for x in exact}
    for c in frame.columns:
        if norm(c) in exact_norm: return c
    for c in frame.columns:
        if any(re.search(p,norm(c)) for p in patterns): return c
    return None
def binary_label(v):
    s=norm(stable_value(v))
    if s in {'1','1_0'} or 'malignant' in s or s in {'malign','cancer'}: return 1
    if s in {'0','0_0'} or 'benign' in s: return 0
    return None
def canonical_pair_key(v):
    s=norm(path_stem(v)); s=re.sub(r'^(mask|bus)_','',s); s=re.sub(r'_(anno|annotation|tumor|mask)$','',s); return s
def match_table_rows(images,frame,id_col):
    index={}; ambiguous=set()
    for idx,val in frame[id_col].items():
        for k in variants(val):
            if k in index and index[k]!=idx: ambiguous.add(k)
            else: index[k]=idx
    out={}
    for v in images:
        keys=list(variants(path_leaf(v))); direct=compact(path_stem(v))
        keys=[direct]+[x for x in keys if x!=direct]
        hits={index[k] for k in keys if k in index and k not in ambiguous}
        if len(hits)==1: out[v]=next(iter(hits))
    return out

rows=[]; statuses=[]
def append_row(d,image_path,mask_path,sample_key,group_raw,released,binary,status,label_basis,group_basis):
    rows.append({'dataset_id':d,'roster_priority':DATASETS.index(d)+1,'sample_id':d+'::'+sample_key,'image_virtual_path':image_path,'mask_virtual_path':mask_path,'released_label':released,'binary_label':binary,'endpoint_status':status,'group_id':d+'::'+stable_value(group_raw),'group_raw':stable_value(group_raw),'label_basis':label_basis,'group_basis':group_basis})

for d in DATASETS:
    try:
        before=len(rows)
        if d=='BUS_BRA_2024':
            images=[v for v in image_entries(d) if is_dir(v,'Images')]; masks=[v for v in image_entries(d) if is_dir(v,'Masks')]
            tables=read_table_entry(d,lambda v:v.lower().endswith('bus_data.csv')); assert len(tables)==1
            _,_,t=tables[0]; idc=pick_col(t,[],('ID',)); casec=pick_col(t,[],('Case',)); labc=pick_col(t,[],('Pathology',)); assert all(x is not None for x in [idc,casec,labc])
            join=match_table_rows(images,t,idc); assert len(images)==1875 and len(join)==1875
            maskmap={canonical_pair_key(v):v for v in masks}
            for v in images:
                r=t.loc[join[v]]; y=binary_label(r[labc]); assert y in {0,1}
                k=canonical_pair_key(v); assert k in maskmap
                append_row(d,v,maskmap[k],k,r[casec],stable_value(r[labc]),y,'INCLUDE_BINARY_LESION','released BUSBRA bus_data.csv Pathology','released BUSBRA bus_data.csv Case')
        elif d=='BUSI_WHU_2025_V3':
            images=[v for v in image_entries(d) if is_dir(v,'img')]; masks=[v for v in image_entries(d) if is_dir(v,'gt')]
            tables=read_table_entry(d,lambda v:v.lower().endswith('patient_infos_busi-whu.xlsx')); assert len(tables)>=1
            t=max((x[2] for x in tables),key=len); idc=pick_col(t,[],('img_names',)); groupc=pick_col(t,[],('patient_index',)); labc=pick_col(t,[r'benign.*malignant'],()); assert all(x is not None for x in [idc,groupc,labc])
            join=match_table_rows(images,t,idc); assert len(images)==927 and len(join)==927
            maskmap={canonical_pair_key(v):v for v in masks}
            for v in images:
                r=t.loc[join[v]]; y=binary_label(r[labc]); assert y in {0,1}
                k=canonical_pair_key(v); assert k in maskmap
                append_row(d,v,maskmap[k],k,r[groupc],stable_value(r[labc]),y,'INCLUDE_BINARY_LESION','released BUSI-WHU workbook benign(0)/malignant(1)','released BUSI-WHU workbook patient_index')
        elif d=='BREAST_LESIONS_USG_2024':
            all_images=image_entries(d); images=[v for v in all_images if not re.search(r'_tumou?r\.',v,flags=re.I)]; masks=[v for v in all_images if re.search(r'_tumou?r\.',v,flags=re.I)]
            xlsx=next(iter(sorted((RECEIPT_ROOT/FOLDERS[d]).glob('*.xlsx')))); book=pd.ExcelFile(xlsx); sheets=[pd.read_excel(xlsx,sheet_name=sh) for sh in book.sheet_names]
            candidates=[]
            for t in sheets:
                idc=pick_col(t,[r'case.*id',r'patient.*id'],('CaseID',)); labc=pick_col(t,[r'diagnos',r'pathol'],('Diagnosis',))
                if idc is not None and labc is not None: candidates.append((t,idc,labc))
            assert candidates; t,idc,labc=max(candidates,key=lambda x:len(x[0])); join=match_table_rows(images,t,idc)
            assert len(images)==256 and len(join)==256
            maskmap={canonical_pair_key(v):v for v in masks}
            for v in images:
                r=t.loc[join[v]]; y=binary_label(r[labc]); assert y in {0,1}
                k=canonical_pair_key(v); assert k in maskmap
                append_row(d,v,maskmap[k],k,r[idc],stable_value(r[labc]),y,'INCLUDE_BINARY_LESION','released TCIA clinical workbook Diagnosis','released TCIA clinical workbook CaseID')
        elif d=='BUS_UCLM_2025_V3':
            images=[v for v in image_entries(d) if is_dir(v,'images')]; masks=[v for v in image_entries(d) if is_dir(v,'masks')]
            maskmap={path_leaf(v):v for v in masks}; assert len(images)==683 and len(maskmap)==683
            counts={'benign':0,'malignant':0,'normal':0}
            for v in images:
                name=path_leaf(v); assert name in maskmap
                arr=np.asarray(Image.open(io.BytesIO(read_virtual(d,maskmap[name]))).convert('RGB'))
                red=bool(np.any((arr[...,0]>200)&(arr[...,1]<80)&(arr[...,2]<80))); green=bool(np.any((arr[...,1]>200)&(arr[...,0]<80)&(arr[...,2]<80)))
                assert not (red and green)
                released='malignant' if red else ('benign' if green else 'normal'); counts[released]+=1
                y=1 if released=='malignant' else (0 if released=='benign' else None); status='INCLUDE_BINARY_LESION' if y is not None else 'EXCLUDE_NORMAL'
                stem=path_stem(v); append_row(d,v,maskmap[name],norm(stem),stem[:4],released,y,status,'released BUS-UCLM RGB mask semantics','first four filename characters per author code')
            assert counts=={'benign':174,'malignant':90,'normal':419}
        elif d=='RODRIGUES_BUI_2017':
            images=[v for v in image_entries(d) if is_dir(v,'originals')]
            assert len(images)==250
            for v in images:
                low=v.lower(); released='malignant' if '/malignant/' in low else ('benign' if '/benign/' in low else '')
                assert released; y=1 if released=='malignant' else 0; stem=path_stem(v)
                append_row(d,v,'',norm(stem),stem,released,y,'INCLUDE_BINARY_LESION','released class-bearing path','released per-image lesion key; patient key unavailable')
        made=pd.DataFrame(rows[before:]); observed=len(made); included=int((made.endpoint_status=='INCLUDE_BINARY_LESION').sum()); groups=int(made.group_id.nunique())
        expected_group_ok=(groups==EXPECTED_GROUPS[d]) if d!='BUS_UCLM_2025_V3' else (groups==38)
        statuses.append({'dataset_id':d,'status':'ADAPTER_PASS','original_images':observed,'endpoint_included':included,'endpoint_excluded':observed-included,'groups_before_endpoint_filter':groups,'expected_originals_match':observed==EXPECTED_ORIGINALS[d],'expected_groups_match':expected_group_ok,'reason':''})
    except Exception as exc:
        rows=rows[:before]
        statuses.append({'dataset_id':d,'status':'HOLD_ADAPTER_SCHEMA','original_images':0,'endpoint_included':0,'endpoint_excluded':0,'groups_before_endpoint_filter':0,'expected_originals_match':False,'expected_groups_match':False,'reason':type(exc).__name__+': '+str(exc)[:500]})

adapter_status=pd.DataFrame(statuses); base_manifest=pd.DataFrame(rows)
if REPLAY: assert canon(pd.read_csv(ADAPTER_STATUS))==canon(adapter_status)
else: write_csv(ADAPTER_STATUS,adapter_status)
display(adapter_status)
assert len(base_manifest)>0,'No dataset adapter produced rows'


,dataset_id,status,original_images,endpoint_included,endpoint_excluded,groups_before_endpoint_filter,expected_originals_match,expected_groups_match,reason
0,BUS_BRA_2024,ADAPTER_PASS,1875,1875,0,1064,True,True,
1,BUSI_WHU_2025_V3,ADAPTER_PASS,927,927,0,816,True,True,
2,BREAST_LESIONS_USG_2024,HOLD_ADAPTER_SCHEMA,0,0,0,0,False,False,AssertionError:
3,BUS_UCLM_2025_V3,ADAPTER_PASS,683,264,419,38,True,True,
4,RODRIGUES_BUI_2017,ADAPTER_PASS,250,250,0,250,True,True,


In [3]:
# @title 11D-R-2. Hash source bytes, decoded pixels, and outcome-free perceptual fingerprints
thumbs={}; hash_rows=[]
for i,r in enumerate(base_manifest.itertuples(index=False),1):
    b=read_virtual(r.dataset_id,r.image_virtual_path)
    with Image.open(io.BytesIO(b)) as im0:
        im=ImageOps.exif_transpose(im0).convert('RGB'); w,h=im.size; rgb=np.asarray(im,dtype=np.uint8)
        header=json.dumps({'width':w,'height':h,'mode':'RGB'},sort_keys=True,separators=(',',':')).encode()+b'\0'
        pixel_sha=sha_bytes(header+rgb.tobytes(order='C'))
        gray=np.asarray(im.convert('L').resize((32,32),Image.Resampling.LANCZOS),dtype=np.float32)
        low=dctn(gray,type=2,norm='ortho')[:8,:8]; med=float(np.median(low.flatten()[1:])); bits=(low.flatten()>med).astype(np.uint8)
        ph=0
        for bit in bits: ph=(ph<<1)|int(bit)
        thumb=np.asarray(im.convert('L').resize((64,64),Image.Resampling.BILINEAR),dtype=np.float32).reshape(-1)
        thumb=(thumb-thumb.mean())/(thumb.std()+1e-8); thumbs[r.sample_id]=thumb
    z=dict(r._asdict()); z.update({'raw_image_sha256':sha_bytes(b),'pixel_sha256':pixel_sha,'pixel_width':w,'pixel_height':h,'pixel_mode':'RGB','phash64':f'{ph:016x}','decode_ok':True})
    hash_rows.append(z)
    if i%500==0: print('Hashed',i,'/',len(base_manifest))
all_manifest=pd.DataFrame(hash_rows).sort_values(['roster_priority','dataset_id','sample_id']).reset_index(drop=True)
assert all_manifest.sample_id.is_unique and all_manifest.decode_ok.all()
if REPLAY: assert canon(pd.read_csv(ALL_MANIFEST))==canon(all_manifest)
else: write_csv(ALL_MANIFEST,all_manifest)
print('Hashed source images:',len(all_manifest),'endpoint included:',int((all_manifest.endpoint_status=='INCLUDE_BINARY_LESION').sum()))


Hashed 500 / 3735
Hashed 1000 / 3735
Hashed 1500 / 3735
Hashed 2000 / 3735
Hashed 2500 / 3735
Hashed 3000 / 3735
Hashed 3500 / 3735
Hashed source images: 3735 endpoint included: 3316


In [4]:
# @title 11D-R-3. Resolve exact duplicates and conservatively quarantine high-confidence derived-copy candidates
work=all_manifest.copy(); work['dedup_status']=np.where(work.endpoint_status=='INCLUDE_BINARY_LESION','KEEP_UNIQUE','EXCLUDE_ENDPOINT')
work['dedup_cluster_id']=''; exact_rows=[]
eligible=work[work.endpoint_status=='INCLUDE_BINARY_LESION']
for pixel_sha,g in eligible.groupby('pixel_sha256',sort=True):
    if len(g)<2: continue
    g=g.sort_values(['roster_priority','dataset_id','sample_id']); labels=sorted(g.binary_label.astype(int).unique().tolist()); cid='EXACT::'+pixel_sha[:16]
    if len(labels)>1:
        action='QUARANTINE_ALL_LABEL_CONFLICT'; keeper=''; work.loc[g.index,'dedup_status']='QUARANTINE_EXACT_LABEL_CONFLICT'
    else:
        keeper=str(g.iloc[0].sample_id); action='KEEP_FROZEN_PRIORITY_CANONICAL'; losers=g.iloc[1:].index; work.loc[losers,'dedup_status']='EXCLUDE_EXACT_PIXEL_DUPLICATE'
    work.loc[g.index,'dedup_cluster_id']=cid
    for r in g.itertuples(): exact_rows.append({'cluster_id':cid,'pixel_sha256':pixel_sha,'sample_id':r.sample_id,'dataset_id':r.dataset_id,'group_id':r.group_id,'binary_label':int(r.binary_label),'action':action,'keeper_sample_id':keeper})
exact_clusters=pd.DataFrame(exact_rows,columns=['cluster_id','pixel_sha256','sample_id','dataset_id','group_id','binary_label','action','keeper_sample_id'])

retained=work[work.dedup_status=='KEEP_UNIQUE'].copy(); near_pairs=[]
def compare_blocks(a,b,same_dataset=False):
    A=list(a.itertuples()); B=list(b.itertuples())
    for i,ra in enumerate(A):
        start=i+1 if same_dataset else 0; ha=int(ra.phash64,16)
        for rb in B[start:]:
            if same_dataset and ra.group_id==rb.group_id: continue
            dist=(ha^int(rb.phash64,16)).bit_count()
            if dist>PHASH_MAX_DISTANCE: continue
            corr=float(np.dot(thumbs[ra.sample_id],thumbs[rb.sample_id])/len(thumbs[ra.sample_id]))
            if corr>=NEAR_COPY_MIN_CORRELATION:
                near_pairs.append({'sample_id_a':ra.sample_id,'dataset_a':ra.dataset_id,'group_a':ra.group_id,'label_a':int(ra.binary_label),'path_a':ra.image_virtual_path,'sample_id_b':rb.sample_id,'dataset_b':rb.dataset_id,'group_b':rb.group_id,'label_b':int(rb.binary_label),'path_b':rb.image_virtual_path,'phash_distance':dist,'thumbnail_correlation':round(corr,6)})

for d in DATASETS:
    x=retained[retained.dataset_id==d]; compare_blocks(x,x,True)
for i,da in enumerate(DATASETS):
    for db in DATASETS[i+1:]: compare_blocks(retained[retained.dataset_id==da],retained[retained.dataset_id==db],False)
near_candidates=pd.DataFrame(near_pairs,columns=['sample_id_a','dataset_a','group_a','label_a','path_a','sample_id_b','dataset_b','group_b','label_b','path_b','phash_distance','thumbnail_correlation'])

if len(near_candidates):
    parent={x:x for x in set(near_candidates.sample_id_a)|set(near_candidates.sample_id_b)}
    def find(x):
        while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]
        return x
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b: parent[max(a,b)]=min(a,b)
    for r in near_candidates.itertuples(): union(r.sample_id_a,r.sample_id_b)
    clusters={}
    for x in parent: clusters.setdefault(find(x),[]).append(x)
    byid=work.set_index('sample_id')
    for root,members in sorted(clusters.items()):
        g=byid.loc[members].sort_values(['roster_priority','dataset_id','sample_id']); labels=g.binary_label.astype(int).unique()
        cid='NEAR::'+sha_bytes('|'.join(sorted(members)).encode())[:16]
        if len(labels)>1: losers=g.index.tolist(); status='QUARANTINE_NEAR_COPY_LABEL_CONFLICT'
        else: losers=g.index.tolist()[1:]; status='EXCLUDE_HIGH_CONFIDENCE_NEAR_COPY'
        work.loc[work.sample_id.isin(losers),'dedup_status']=status; work.loc[work.sample_id.isin(members),'dedup_cluster_id']=cid

frozen=work[work.dedup_status=='KEEP_UNIQUE'].copy().sort_values(['roster_priority','dataset_id','sample_id']).reset_index(drop=True)
summary=[]
for d in DATASETS:
    x=work[work.dataset_id==d]; y=frozen[frozen.dataset_id==d]
    summary.append({'dataset_id':d,'all_source_images':len(x),'endpoint_eligible_before_dedup':int((x.endpoint_status=='INCLUDE_BINARY_LESION').sum()),'endpoint_excluded':int((x.endpoint_status!='INCLUDE_BINARY_LESION').sum()),'exact_duplicate_or_conflict_retired':int(x.dedup_status.isin(['EXCLUDE_EXACT_PIXEL_DUPLICATE','QUARANTINE_EXACT_LABEL_CONFLICT']).sum()),'high_confidence_near_copy_retired':int(x.dedup_status.isin(['EXCLUDE_HIGH_CONFIDENCE_NEAR_COPY','QUARANTINE_NEAR_COPY_LABEL_CONFLICT']).sum()),'retained_binary_images':len(y),'retained_groups':int(y.group_id.nunique()),'retained_negative':int((y.binary_label==0).sum()),'retained_positive':int((y.binary_label==1).sum())})
dedup_summary=pd.DataFrame(summary)
if REPLAY:
    assert canon(pd.read_csv(EXACT_CLUSTERS))==canon(exact_clusters); assert canon(pd.read_csv(NEAR_CANDIDATES))==canon(near_candidates); assert canon(pd.read_csv(DEDUP_SUMMARY))==canon(dedup_summary); assert canon(pd.read_csv(EXACT_MANIFEST))==canon(frozen)
else:
    write_csv(EXACT_CLUSTERS,exact_clusters); write_csv(NEAR_CANDIDATES,near_candidates); write_csv(DEDUP_SUMMARY,dedup_summary); write_csv(EXACT_MANIFEST,frozen)
display(dedup_summary); print('Exact duplicate rows:',len(exact_clusters),'high-confidence near-copy pairs:',len(near_candidates),'retained:',len(frozen))


,dataset_id,all_source_images,endpoint_eligible_before_dedup,endpoint_excluded,exact_duplicate_or_conflict_retired,high_confidence_near_copy_retired,retained_binary_images,retained_groups,retained_negative,retained_positive
0,BUS_BRA_2024,1875,1875,0,0,0,1875,1064,1268,607
1,BUSI_WHU_2025_V3,927,927,0,0,1,926,815,559,367
2,BREAST_LESIONS_USG_2024,0,0,0,0,0,0,0,0,0
3,BUS_UCLM_2025_V3,683,264,419,1,0,263,36,174,89
4,RODRIGUES_BUI_2017,250,250,0,0,8,242,242,96,146


Exact duplicate rows: 2 high-confidence near-copy pairs: 9 retained: 3306


In [5]:
# @title 11D-R-4. Freeze deterministic group-disjoint held-out and OOF partitions
def hkey(*parts): return hashlib.sha256('|'.join(map(str,parts)).encode()).hexdigest()
def group_table(x):
    out=[]
    for g,z in x.groupby('group_id',sort=True):
        n0=int((z.binary_label==0).sum()); n1=int((z.binary_label==1).sum()); stratum='mixed' if n0 and n1 else ('positive' if n1 else 'negative')
        out.append({'group_id':g,'negative':n0,'positive':n1,'images':len(z),'stratum':stratum})
    return pd.DataFrame(out)
def choose_holdout(gt,d):
    best=None
    for trial in range(256):
        chosen=[]
        for s,z in gt.groupby('stratum',sort=True):
            ordered=sorted(z.group_id,key=lambda g:hkey(SEED+trial,d,g))
            n=max(1,int(round(HOLDOUT_FRACTION*len(ordered)))) if len(ordered)>=2 else 0
            chosen+=ordered[:n]
        hold=gt[gt.group_id.isin(chosen)]; dev=gt[~gt.group_id.isin(chosen)]
        valid=len(hold)>0 and len(dev)>=MIN_GROUPS and hold.negative.sum()>0 and hold.positive.sum()>0 and dev.negative.sum()>0 and dev.positive.sum()>0
        pg0=int((dev.negative>0).sum()); pg1=int((dev.positive>0).sum()); valid=valid and min(pg0,pg1)>=MIN_OOF_FOLDS
        if not valid: continue
        overall=gt.positive.sum()/max(gt.negative.sum()+gt.positive.sum(),1); hp=hold.positive.sum()/max(hold.negative.sum()+hold.positive.sum(),1)
        score=abs(len(hold)/len(gt)-HOLDOUT_FRACTION)+abs(hp-overall)
        key=(round(score,12),trial,sorted(chosen))
        if best is None or key<best[0]: best=(key,set(chosen))
    if best is None: raise ValueError('No valid two-class group-disjoint held-out split')
    return best[1]
def choose_folds(dev_gt,d,k):
    best=None
    for trial in range(256):
        assignment={}
        for s,z in dev_gt.groupby('stratum',sort=True):
            ordered=sorted(z.group_id,key=lambda g:hkey(SEED+1000+trial,d,s,g))
            for i,g in enumerate(ordered): assignment[g]=i%k
        q=dev_gt.copy(); q['fold']=q.group_id.map(assignment); valid=all(q[q.fold==f].negative.sum()>0 and q[q.fold==f].positive.sum()>0 for f in range(k))
        if not valid: continue
        counts=q.groupby('fold')[['negative','positive','images']].sum(); target=counts.sum()/k; score=float((((counts-target)/(target+1e-8))**2).to_numpy().sum())
        key=(round(score,12),trial,sorted(assignment.items()))
        if best is None or key<best[0]: best=(key,assignment)
    if best is None: raise ValueError('No valid two-class OOF fold assignment')
    return best[1]

group_rows=[]; split_rows=[]; split_summary=[]; split_ready=[]
for d in DATASETS:
    x=frozen[frozen.dataset_id==d].copy()
    try:
        gt=group_table(x); assert len(gt)>=MIN_GROUPS and gt.negative.sum()>0 and gt.positive.sum()>0
        hold=choose_holdout(gt,d); dev_gt=gt[~gt.group_id.isin(hold)].copy(); k=min(MAX_OOF_FOLDS,int((dev_gt.negative>0).sum()),int((dev_gt.positive>0).sum())); assert k>=MIN_OOF_FOLDS
        folds=choose_folds(dev_gt,d,k)
        for r in gt.itertuples(): group_rows.append({'dataset_id':d,'group_id':r.group_id,'stratum':r.stratum,'negative_images':r.negative,'positive_images':r.positive,'images':r.images,'partition':'heldout' if r.group_id in hold else 'development','oof_fold':-1 if r.group_id in hold else int(folds[r.group_id]),'split_seed':SEED})
        ga=pd.DataFrame([r for r in group_rows if r['dataset_id']==d]); xm=x.merge(ga[['group_id','partition','oof_fold','split_seed']],on='group_id',how='left',validate='many_to_one'); split_rows.extend(xm.to_dict('records'))
        split_ready.append(d)
        for part,z in xm.groupby('partition',sort=True): split_summary.append({'dataset_id':d,'status':'SPLIT_READY','partition':part,'oof_fold':-1,'images':len(z),'groups':z.group_id.nunique(),'negative':int((z.binary_label==0).sum()),'positive':int((z.binary_label==1).sum()),'oof_folds':k,'reason':''})
        for f,z in xm[xm.partition=='development'].groupby('oof_fold',sort=True): split_summary.append({'dataset_id':d,'status':'SPLIT_READY','partition':'development_oof','oof_fold':int(f),'images':len(z),'groups':z.group_id.nunique(),'negative':int((z.binary_label==0).sum()),'positive':int((z.binary_label==1).sum()),'oof_folds':k,'reason':''})
    except Exception as exc:
        split_summary.append({'dataset_id':d,'status':'HOLD_SPLIT_NOT_FEASIBLE','partition':'','oof_fold':-1,'images':len(x),'groups':x.group_id.nunique(),'negative':int((x.binary_label==0).sum()) if len(x) else 0,'positive':int((x.binary_label==1).sum()) if len(x) else 0,'oof_folds':0,'reason':type(exc).__name__+': '+str(exc)[:500]})
group_assign=pd.DataFrame(group_rows,columns=['dataset_id','group_id','stratum','negative_images','positive_images','images','partition','oof_fold','split_seed'])
split_manifest=pd.DataFrame(split_rows)
split_summary=pd.DataFrame(split_summary)
source_authorised=len(split_ready)>=MINIMUM_SPLIT_READY_DOMAINS
decision='SEAL_STAGE11D_R_AUTHORISE_STAGE11E_R_DEVELOPMENT_ONLY_SOURCE_RECOVERABILITY_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED' if source_authorised else 'HOLD_STAGE11D_R_INSUFFICIENT_SPLIT_READY_DOMAINS_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED'
if REPLAY:
    assert canon(pd.read_csv(GROUP_ASSIGN))==canon(group_assign); assert canon(pd.read_csv(SPLIT_MANIFEST))==canon(split_manifest); assert canon(pd.read_csv(SPLIT_SUMMARY))==canon(split_summary); handoff=verify_self(HANDOFF,'handoff_sha256')
else:
    write_csv(GROUP_ASSIGN,group_assign); write_csv(SPLIT_MANIFEST,split_manifest); write_csv(SPLIT_SUMMARY,split_summary)
    handoff={'stage':'Stage11D-R','target':'Stage11E-R','decision':decision,'protocol_seal_sha256':seal['seal_sha256'],'parent_stage11c_r2_final_sha256':r2['final_record_sha256'],'exact_manifest_sha256':sha_file(EXACT_MANIFEST),'dedup_summary_sha256':sha_file(DEDUP_SUMMARY),'group_assignment_sha256':sha_file(GROUP_ASSIGN),'split_manifest_sha256':sha_file(SPLIT_MANIFEST),'split_ready_dataset_ids':split_ready,'split_ready_domain_count':len(split_ready),'minimum_split_ready_domains':MINIMUM_SPLIT_READY_DOMAINS,'source_recoverability_authorised':source_authorised,'source_gate':'grouped development OOF and separate grouped held-out AUC each >=0.70 with group-bootstrap lower 95% CI >0.55','minimum_recoverable_sources_for_global_edge_gate':3,'stage12_authorised':False,'ddo2_fitted':False,'locked_blind_assets_touched':False}
    handoff['handoff_sha256']=sha_json(handoff); write_json(HANDOFF,handoff)
display(split_summary); print('Split-ready domains:',split_ready,'Stage11E-R authorised:',source_authorised)


,dataset_id,status,partition,oof_fold,images,groups,negative,positive,oof_folds,reason
0,BUS_BRA_2024,SPLIT_READY,development,-1,1498,852,1013,485,5,
1,BUS_BRA_2024,SPLIT_READY,heldout,-1,377,212,255,122,5,
2,BUS_BRA_2024,SPLIT_READY,development_oof,0,298,171,201,97,5,
3,BUS_BRA_2024,SPLIT_READY,development_oof,1,302,171,204,98,5,
4,BUS_BRA_2024,SPLIT_READY,development_oof,2,301,171,204,97,5,
5,BUS_BRA_2024,SPLIT_READY,development_oof,3,297,170,200,97,5,
6,BUS_BRA_2024,SPLIT_READY,development_oof,4,300,169,204,96,5,
7,BUSI_WHU_2025_V3,SPLIT_READY,development,-1,740,652,447,293,5,
8,BUSI_WHU_2025_V3,SPLIT_READY,heldout,-1,186,163,112,74,5,
9,BUSI_WHU_2025_V3,SPLIT_READY,development_oof,0,147,131,90,57,5,


Split-ready domains: ['BUS_BRA_2024', 'BUSI_WHU_2025_V3', 'BUS_UCLM_2025_V3', 'RODRIGUES_BUI_2017'] Stage11E-R authorised: True


In [6]:
# @title 11D-R-5. Run independent endpoint, duplication, leakage, replay, and firewall checks
checks=[]
def ck(name,passed,evidence): checks.append({'check':name,'passed':bool(passed),'evidence':str(evidence)[:2000]})
ck('sealed Stage10 parent exact',stage10['final_record_sha256']==EXPECTED_STAGE10_FINAL,stage10['final_record_sha256'])
ck('sealed Stage11C-R2 parent exact',r2['final_record_sha256']==EXPECTED_R2_FINAL,r2['final_record_sha256'])
ck('Stage11C-R2 handoff exact',r2_handoff['handoff_sha256']==EXPECTED_R2_HANDOFF,r2_handoff['handoff_sha256'])
ck('five-domain roster exact',roster.dataset_id.tolist()==DATASETS,roster.dataset_id.tolist())
ck('receipt bytes unchanged',observed_receipts.matches_stage11c_r2.all(),observed_receipts.sha256.tolist())
ck('all successful adapters exact count',all((r.status!='ADAPTER_PASS') or (r.expected_originals_match and r.expected_groups_match) for r in adapter_status.itertuples()),adapter_status[['dataset_id','status','original_images','groups_before_endpoint_filter']].to_dict('records'))
ck('normal excluded from binary endpoint',not ((all_manifest.released_label.astype(str).str.lower()=='normal')&(all_manifest.endpoint_status=='INCLUDE_BINARY_LESION')).any(),int((all_manifest.released_label.astype(str).str.lower()=='normal').sum()))
ck('binary endpoint labels exact',set(frozen.binary_label.astype(int).unique()).issubset({0,1}),sorted(frozen.binary_label.astype(int).unique().tolist()))
ck('sample identifiers unique',all_manifest.sample_id.is_unique,len(all_manifest))
ck('no mask used as source image',not all_manifest.image_virtual_path.str.contains(r'(?:^|[/!])(?:masks?|gt)(?:[/!]|$)|_tumou?r\.',case=False,regex=True).any(),'source paths audited')
ck('retained pixel hashes unique',frozen.pixel_sha256.is_unique,len(frozen))
ck('exact duplicate actions complete',not work[(work.endpoint_status=='INCLUDE_BINARY_LESION')&(work.dedup_status=='KEEP_UNIQUE')].pixel_sha256.duplicated().any(),len(exact_clusters))
ck('high-confidence near candidates resolved',all(x not in set(frozen.sample_id) or y not in set(frozen.sample_id) for x,y in near_candidates[['sample_id_a','sample_id_b']].itertuples(index=False,name=None)) if len(near_candidates) else True,len(near_candidates))
ck('group appears in one partition',group_assign.groupby(['dataset_id','group_id']).partition.nunique().max()<=1 if len(group_assign) else True,len(group_assign))
ck('development group appears in one OOF fold',group_assign[group_assign.partition=='development'].groupby(['dataset_id','group_id']).oof_fold.nunique().max()<=1 if len(group_assign) else True,len(group_assign))
ck('heldout has no OOF fold',(group_assign.loc[group_assign.partition=='heldout','oof_fold']==-1).all() if len(group_assign) else True,'-1')
ck('each split-ready partition has both classes',all(z.binary_label.nunique()==2 for _,z in split_manifest.groupby(['dataset_id','partition'])) if len(split_manifest) else False,split_ready)
inventory_text='|'.join(all_manifest.image_virtual_path.astype(str).tolist()).lower()
ck('no locked-blind token in accessed image paths',not any(x.lower() in inventory_text for x in LOCKED),LOCKED)
ck('no performance input',runtime['performance_evaluated'] is False and runtime['embeddings_computed'] is False,'false')
ck('source axes and transfer edges prohibited',runtime['source_axes_fitted'] is False and runtime['transfer_edges_evaluated'] is False,'false')
ck('DDO2 and Stage12 prohibited',runtime['ddo2_fitted'] is False and handoff['stage12_authorised'] is False,'false')
ck('locked blind untouched',runtime['locked_blind_assets_touched'] is False and handoff['locked_blind_assets_touched'] is False,'false')
ck('handoff self hash',sha_json({k:v for k,v in handoff.items() if k!='handoff_sha256'})==handoff['handoff_sha256'],handoff['handoff_sha256'])
validity=pd.DataFrame(checks)
if REPLAY: assert canon(pd.read_csv(FIREWALL))==canon(validity)
else: write_csv(FIREWALL,validity)
failed=validity.loc[~validity.passed,'check'].tolist(); assert not failed,'Validity failure: '+'; '.join(failed)
print('Independent checks:',int(validity.passed.sum()),'/',len(validity),'passed')


Independent checks: 23 / 23 passed


In [7]:
# @title 11D-R-6. Seal report, integrity manifest, final record, and print the next decision gate
potential_edges=len(split_ready)*(len(split_ready)-1); minimum_new_edges=3*(len(split_ready)-1) if len(split_ready)>=4 else 0; projected_total=21+minimum_new_edges
report=f"""# Stage 11D-R report

## Answer first

- Qualified input domains: **{len(DATASETS)}**.
- Source images mapped before endpoint filtering: **{len(all_manifest)}**.
- Binary lesion images before deduplication: **{int((all_manifest.endpoint_status=='INCLUDE_BINARY_LESION').sum())}**.
- Exact duplicate-cluster rows: **{len(exact_clusters)}**.
- High-confidence near-copy candidate pairs: **{len(near_candidates)}**; all candidate clusters were conservatively resolved before splitting.
- Retained binary images after endpoint filtering and deduplication: **{len(frozen)}**.
- Split-ready domains: **{len(split_ready)}/{MINIMUM_SPLIT_READY_DOMAINS}** — {', '.join(split_ready) if split_ready else 'none'}.
- Stage 11E-R development-only source recoverability authorised: **{source_authorised}**.
- Decision: `{decision}`.

## Deduplication and endpoint summary

{markdown_table(dedup_summary)}

## Grouped split summary

{markdown_table(split_summary)}

## Capacity implication

The split-ready roster contains {len(split_ready)} domains and therefore at most {potential_edges} directed within-modality development edges. The frozen global gate still requires at least three recoverable sources. If exactly three sources pass, they contribute {minimum_new_edges} new directed edges and raise the existing 21-edge library to {projected_total}. This is a capacity projection only; no source or edge performance was computed here.

## Method boundary

Stage 11D-R used only released labels/groups and image content required for duplicate control. It did not compute embeddings, source axes, AUCs, transfer results, thresholds, DDO2 coefficients, Stage 12 results, or any locked-blind result. Rodrigues remains lesion-grouped rather than patient-grouped because no released patient key was available; that limitation remains explicit in the frozen manifest and later report.
"""
if REPLAY: assert REPORT.read_text(encoding='utf-8')==report
else: write_text(REPORT,report)
tracked=[PROTOCOL,PARENT_COMMIT,ADAPTER_STATUS,ALL_MANIFEST,EXACT_MANIFEST,EXACT_CLUSTERS,NEAR_CANDIDATES,DEDUP_SUMMARY,GROUP_ASSIGN,SPLIT_MANIFEST,SPLIT_SUMMARY,HANDOFF,FIREWALL,REPORT]
manifest=pd.DataFrame([{'relative_path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha_file(p)} for p in tracked])
if REPLAY:
    old=pd.read_csv(OUTPUT_MANIFEST); assert canon(old)==canon(manifest)
    for r in old.itertuples():
        p=ROOT/r.relative_path; assert p.is_file() and p.stat().st_size==int(r.size_bytes) and sha_file(p)==r.sha256
else: write_csv(OUTPUT_MANIFEST,manifest)
next_step='BUILD_STAGE11E_R_DEVELOPMENT_ONLY_SOURCE_RECOVERABILITY_FROM_FROZEN_GROUPED_MANIFESTS' if source_authorised else 'REVIEW_STAGE11D_R_TYPED_ADAPTER_OR_SPLIT_HOLDS_WITHOUT_PERFORMANCE_BASED_SUBSTITUTION'
payload={'stage':'Stage11D-R','version':'0.1','decision':decision,'protocol_seal_sha256':seal['seal_sha256'],'parent_stage11c_r2_final_sha256':r2['final_record_sha256'],'parent_stage11c_r2_handoff_sha256':r2_handoff['handoff_sha256'],'all_manifest_sha256':sha_file(ALL_MANIFEST),'exact_manifest_sha256':sha_file(EXACT_MANIFEST),'exact_duplicate_cluster_sha256':sha_file(EXACT_CLUSTERS),'near_copy_candidate_sha256':sha_file(NEAR_CANDIDATES),'dedup_summary_sha256':sha_file(DEDUP_SUMMARY),'group_assignment_sha256':sha_file(GROUP_ASSIGN),'split_manifest_sha256':sha_file(SPLIT_MANIFEST),'split_summary_sha256':sha_file(SPLIT_SUMMARY),'stage11e_r_handoff_sha256':handoff['handoff_sha256'],'output_integrity_manifest_sha256':sha_file(OUTPUT_MANIFEST),'source_images_mapped':len(all_manifest),'binary_images_before_dedup':int((all_manifest.endpoint_status=='INCLUDE_BINARY_LESION').sum()),'retained_binary_images':len(frozen),'exact_duplicate_cluster_rows':len(exact_clusters),'near_copy_candidate_pairs':len(near_candidates),'split_ready_domains':len(split_ready),'minimum_split_ready_domains':MINIMUM_SPLIT_READY_DOMAINS,'source_recoverability_authorised':source_authorised,'stage12_authorised':False,'ddo2_fitted':False,'locked_blind_assets_touched':False,'next_step':next_step}
if REPLAY:
    final=verify_self(FINAL,'final_record_sha256')
    for k,v in payload.items(): assert final[k]==v,f'Replay final changed: {k}'
else:
    final=dict(payload); final['completed_utc']=now(); final['final_record_sha256']=sha_json(final); write_json(FINAL,final)
runtime.update({'completed':True,'decision':decision,'final_record_sha256':final['final_record_sha256'],'last_updated_utc':now()}); RUNTIME.write_text(json.dumps(runtime,indent=2)+'\n')
print('================ STAGE 11D-R COMPLETE ================')
print('Mapped source images / binary before dedup / retained:',final['source_images_mapped'],'/',final['binary_images_before_dedup'],'/',final['retained_binary_images'])
print('Exact duplicate cluster rows / near-copy candidate pairs:',final['exact_duplicate_cluster_rows'],'/',final['near_copy_candidate_pairs'])
print('Split-ready development domains / minimum:',f"{final['split_ready_domains']}/{final['minimum_split_ready_domains']}")
print('Stage11E-R source recoverability authorised:',final['source_recoverability_authorised'])
print('Decision:',final['decision'])
print('Protocol seal:',final['protocol_seal_sha256'])
print('Exact manifest hash:',final['exact_manifest_sha256'])
print('Dedup summary hash:',final['dedup_summary_sha256'])
print('Group assignment hash:',final['group_assignment_sha256'])
print('Grouped split manifest hash:',final['split_manifest_sha256'])
print('Stage11E-R handoff hash:',final['stage11e_r_handoff_sha256'])
print('Final record hash:',final['final_record_sha256'])
print('Next step:',final['next_step'])


================ STAGE 11D-R COMPLETE ================
Mapped source images / binary before dedup / retained: 3735 / 3316 / 3306
Exact duplicate cluster rows / near-copy candidate pairs: 2 / 9
Split-ready development domains / minimum: 4/4
Stage11E-R source recoverability authorised: True
Decision: SEAL_STAGE11D_R_AUTHORISE_STAGE11E_R_DEVELOPMENT_ONLY_SOURCE_RECOVERABILITY_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED
Protocol seal: 8f2c409cf51429c33aea93b0c8788484c53c909865c4153ca88de3bb3877cc98
Exact manifest hash: 3e66aad296106d12958a8d84634e5cc780399329ba22b503e49f42019bf3db11
Dedup summary hash: 6e41e091759cfea56bbfb42ec841677d3e0b844a86ab882e486251ec3886df13
Group assignment hash: 7580876ab7872e4de0de1ca9d4b2a35c5f0b04439f6658ed5b8d47f86c9f21c1
Grouped split manifest hash: 9cd17234463ed8d94bbc76965770559a1e61294385b5795a507f1bcff907158d
Stage11E-R handoff hash: e68d4e1ffccb9554c549c29a7af3d490c4a9fbc62e7bf082b3bdb5b6c36d332f
Final record hash: 0e35ab95ad1c087efa43de704bdce11152745b812ef